# Demo: Private Synthetic Text Generation on AGNews

This notebook shows how to run the private prediction algorithm. We use the following configuration

- Dataset: `agnews`
- Privacy budget: `epsilon = 3.0`
- Batch size: `s = 255`
- Clip bound: `c = 10.0`
- Temperature (private tokens): `tau = 2.0`
- Public temperature: `tau_public = 1.5`
- SVT threshold: `theta = 0.5`
- SVT noise: `sigma = 0.2`
- Max private tokens per example: `r = 64`
- Number of source examples used: `num_examples = 1000`

We assume:
- You have already run `scripts/local_scripts/install_amazon.sh` (or otherwise created the `llm_scoring_venv` venv and downloaded the Gemma model), and
- You are running this notebook from the project root (`llm_plus_scoring_synthetic_gen`).

We will:
1. Import the core library functions (`load_dataset_examples`, `load_model`, `generate_synthetic_dataset`, etc.)
2. Configure the privacy and generation parameters in Python
3. Resolve all local paths from `data/` for dataset cache, model, and outputs
4. Run generation and save the output JSONL
5. Inspect metadata and a few synthetic examples.

In [1]:
# 1. Environment check and central paths
import os, sys, json, glob
from pathlib import Path

# This notebook lives in project_root/Notebook/, so resolve paths from here.
notebook_dir = Path.cwd()
project_dir = notebook_dir.parent if notebook_dir.name == "Notebook" else notebook_dir
src_dir = project_dir / "src"
data_dir = project_dir / "data"
dataset_cache_dir = data_dir / "datasets"
model_dir = data_dir / "models" / "gemma-2-2b-it"
output_dir = data_dir / "outputs"

# Add the project root so imports like src.* and scripts.* work.
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))

output_dir.mkdir(parents=True, exist_ok=True)

print("Project dir:", project_dir)
print("Data dir:", data_dir)
print("Dataset cache dir:", dataset_cache_dir)
print("Model dir:", model_dir)
print("Output dir:", output_dir)
print("Python:", sys.version)
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))

Project dir: /home/ubuntu/llm_plus_scoring_synthetic_gen
Data dir: /home/ubuntu/llm_plus_scoring_synthetic_gen/data
Dataset cache dir: /home/ubuntu/llm_plus_scoring_synthetic_gen/data/datasets
Model dir: /home/ubuntu/llm_plus_scoring_synthetic_gen/data/models/gemma-2-2b-it
Output dir: /home/ubuntu/llm_plus_scoring_synthetic_gen/data/outputs
Python: 3.10.12 (main, Nov  4 2025, 08:48:33) [GCC 11.4.0]
CUDA_VISIBLE_DEVICES: None


In [2]:
# 2. Import library functions and define parameters

from src.config import PrivacyConfig, GenerationConfig, ModelConfig
from src.privacy_accounting import privacy_report
from src.generate import generate_synthetic_dataset
from src.evaluate import save_synthetic_data, compute_generation_stats
from scripts.run_experiment import load_dataset_examples, load_model

# Match the CLI arguments we would have passed
DATASET = "agnews"
EPSILON = 3.0
NUM_EXAMPLES = 1000
BATCH_SIZE = 255
CLIP_BOUND = 10.0
TEMPERATURE = 2.0
PUBLIC_TEMPERATURE = 1.5
SVT_THRESHOLD = 0.5
SVT_NOISE = 0.2
MAX_PRIVATE_TOKENS = 64
TOP_K_VOCAB = 1024
DELTA = None  # will default to 1 / n at runtime

print("Configured parameters:")
for k, v in {
    "dataset": DATASET,
    "epsilon": EPSILON,
    "num_examples": NUM_EXAMPLES,
    "batch_size": BATCH_SIZE,
    "clip_bound": CLIP_BOUND,
    "temperature": TEMPERATURE,
    "public_temperature": PUBLIC_TEMPERATURE,
    "svt_threshold": SVT_THRESHOLD,
    "svt_noise": SVT_NOISE,
    "max_private_tokens": MAX_PRIVATE_TOKENS,
    "top_k_vocab": TOP_K_VOCAB,
}.items():
    print(f"  {k}: {v}")

Configured parameters:
  dataset: agnews
  epsilon: 3.0
  num_examples: 1000
  batch_size: 255
  clip_bound: 10.0
  temperature: 2.0
  public_temperature: 1.5
  svt_threshold: 0.5
  svt_noise: 0.2
  max_private_tokens: 64


In [3]:
# 3. Load dataset and configure privacy / generation

# Load a subset of AGNews from HuggingFace, caching under data_dir.
examples = load_dataset_examples(
    DATASET,
    num_examples=NUM_EXAMPLES,
    cache_dir=str(dataset_cache_dir),
)
print(f"Loaded {len(examples)} examples for dataset='{DATASET}'")

# Configure privacy (delta defaults to 1 / n if None)
if DELTA is None:
    DELTA = 1.0 / len(examples)

svt_noise = SVT_NOISE if SVT_THRESHOLD != float("-inf") else None

privacy_config = PrivacyConfig(
    target_epsilon=EPSILON,
    delta=DELTA,
    clip_bound=CLIP_BOUND,
    temperature=TEMPERATURE,
    public_temperature=PUBLIC_TEMPERATURE,
    svt_threshold=SVT_THRESHOLD,
    svt_noise=SVT_NOISE,
)

gen_config = GenerationConfig(
    batch_size=BATCH_SIZE,
    max_private_tokens=MAX_PRIVATE_TOKENS,
    max_total_tokens=256,
    top_k_vocab=1024,
)

# Print privacy report for these settings
report = privacy_report(
    MAX_PRIVATE_TOKENS,
    CLIP_BOUND,
    BATCH_SIZE,
    TEMPERATURE,
    DELTA,
    svt_noise,
)

print("\nPrivacy configuration:")
for k, v in report.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.6f}")
    else:
        print(f"  {k}: {v}")

Loading fancyzhx/ag_news (train split)...
  Loaded 1000 examples with 4 labels
Loaded 1000 examples for dataset='agnews'

Privacy configuration:
  rho_per_token: 0.000961
  total_rho: 0.061515
  epsilon: 1.365247
  delta: 0.001000
  num_private_tokens: 64
  clip_bound: 10.000000
  batch_size: 255
  temperature: 2.000000
  svt_noise: 0.200000


In [4]:
# 4. Load the Gemma model from data_dir

import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model_config = ModelConfig(
    model_name_or_path=str(model_dir),
    device=device,
)

model, tokenizer = load_model(model_config)

print("Model and tokenizer loaded.")
print("Resolved device:", device)

Loading model from /home/ubuntu/llm_plus_scoring_synthetic_gen/data/models/gemma-2-2b-it...


Loading weights:   0%|          | 0/288 [00:00<?, ?it/s]

  Model loaded on cuda:0, dtype=torch.bfloat16
Model and tokenizer loaded.
Resolved device: cuda


In [5]:
# 5. Generate synthetic dataset and save to JSONL under data_dir

from time import time
import time as _time

start = time()
synthetic_data = generate_synthetic_dataset(
    model,
    tokenizer,
    examples,
    dataset_name=DATASET,
    privacy_config=privacy_config,
    gen_config=gen_config,
    text_column="text",
    label_column="label",
    device=model_config.device,
    verbose=True,
)
elapsed = time() - start

print(f"\nGeneration took {elapsed:.1f}s")

timestamp = _time.strftime("%Y%m%d_%H%M%S")
output_path = output_dir / f"{DATASET}_eps{EPSILON}_s{BATCH_SIZE}_{timestamp}.jsonl"

metadata = {
    "dataset": DATASET,
    "epsilon": report["epsilon"],
    "delta": DELTA,
    "batch_size": BATCH_SIZE,
    "clip_bound": CLIP_BOUND,
    "temperature": TEMPERATURE,
    "svt_threshold": SVT_THRESHOLD,
    "svt_noise": SVT_NOISE,
    "max_private_tokens": MAX_PRIVATE_TOKENS,
    "num_source_examples": len(examples),
    "generation_time_seconds": elapsed,
}

save_synthetic_data(synthetic_data, str(output_path), metadata=metadata)
print(f"Saved {len(synthetic_data)} examples to {output_path}")

# Compute and print summary statistics
stats = compute_generation_stats(synthetic_data)
print("\nGeneration statistics:")
for k, v in stats.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

Privacy report: epsilon=1.3652, delta=1.00e-03
  rho/token=0.000961, total_rho=0.061515
  5 batches, batch_size=255

  Batch 1/5 (label=World, size=244)


KeyboardInterrupt: 

In [ ]:
# 6. Reload the saved JSONL and inspect a few examples

latest = output_path
print("Latest output:", latest)

with open(latest) as f:
    lines = f.readlines()

meta = json.loads(lines[0])
print("\nMetadata from file:\n", json.dumps(meta["_metadata"], indent=2))

examples_from_file = [json.loads(l) for l in lines[1:6]]
print("\nSample synthetic examples:")
for i, ex in enumerate(examples_from_file, 1):
    print(f"Example {i} — label={ex['label_name']}")
    print(ex["text"])
    print(f"tokens: private={ex['num_private_tokens']}, public={ex['num_public_tokens']}")
    print("-" * 80)